# SDF+

Determining signed distance to curve and $t$ of closest point.


In [ ]:
%%html
<style>
    :root {
        --jp-content-font-color0: var(--vscode-editor-foreground);
        --jp-content-font-color1: var(--vscode-editor-foreground);
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


In [ ]:
import math
import numpy as np

arr = np.array

In [ ]:
import pyvista as pv
from matplotlib.cm import tab10

# pv.set_jupyter_backend("client")
tab10colors = tab10.colors  # type: ignore
yrnclr = [tab10.colors[4], tab10.colors[6]]

In [ ]:
from splines import M1, M2, mega_curve


def spline1(points):
    return mega_curve(1, M1, points, arr((0.0, 1.0)))

In [ ]:
RES = 64
PIX = 1.0 / RES
X, Y = np.meshgrid(np.arange(RES - 1), np.arange(RES - 1))
PP = (np.column_stack([X.flatten(), Y.flatten()]) + 0.5) * PIX

grid = pv.ImageData(dimensions=(RES, RES, 1), spacing=(PIX, PIX, PIX))

In [ ]:
N = 5
ls = np.linspace(0.0, 1.0, N)
points = arr([
    ls + np.random.random(N) * 0.5 - 0.25,
    ls + np.random.random(N) * 0.5 - 0.25,
    np.zeros(N) + 0.1,
]).T

points2d = points[:, :2]

In [ ]:
pl = pv.Plotter()
pl.background_color = pv.Color("#303030")
pl.add_mesh(pv.lines_from_points(points), color="black", line_width=2)
pl.view_xy()
pl.show()

## Points spline

Just points, distance to them


In [ ]:
def disq0(p1, ps):
    s = ps - p1
    return s @ s


def dist0map(ps):
    d_ = [disq0(points2d[i], ps) for i in range(N)]
    d = min(d_)
    return math.sqrt(d)

In [ ]:
values = np.fromiter(map(dist0map, PP), dtype=float)

In [ ]:
grid.cell_data["scalars"] = values
_ = pl.add_mesh(grid, name="tex", cmap="seismic", clim=[-1.0, +1.0], interpolate_before_map=False)

## Orientation

For anchor vector $V_a$ and sample vector $V_s$ all in $\mathbb{R}^2$

With angles anti-clockwise, and screws right-handed.

- $z_c = V_a \cdot V = |V_a||V_s| cos \phi$
- $=x_a x_s + y_a y_s$

$
\begin{cases}
z_c > 0 \implies |\phi| < 90° \text{— same direction} \\
z_c = 0 \implies |\phi| = 90° \text{— orthogonal}\\
z_c < 0 \implies |\phi| > 90° \text{— opposite direction} \\
\end{cases}
$

- $z_s = (V^*_a \times V^*_s)_z = |V_a||V_s| sin \phi$
- $= x_a y_s - y_a x_s$

$
\begin{cases}
z_s > 0 \implies 0 < \phi < 180° \text{— left-hand side} \\
z_s = 0 \implies \phi = 0° \vee \phi = ±180° \text{— collinear} \\
z_s < 0 \implies -180 < \phi < 0° \text{— right-hand side} \\
\end{cases}
$


In [ ]:
def sideZ(v1, v2):
    return np.sign(v1[0] * v2[1] - v1[1] * v2[0])


def side1(p1, p2, ps):
    p2_ = p2 - p1
    ps_ = ps - p1
    return sideZ(p2_, ps_)

> Problem: for edge points it gives same distance with different sign on adjacent segments


## Linear spline

For a local segment $\big[ P_1, P_2 \big]$ and sample point $P_s$ (localized to $P_0$)

- $P'_1 = P_1 - P_1 = 0$
- $P'_2 = P_2 - P_1$
- $P'_s = P_s - P_1$
- $l = |P_2 - P_1|^2 = P'_2 \cdot P'_2$ — segment squared length
- $j = P'_s \cdot P'_2$ — projlength of sample to segment
- $t = j / l $ — relative proj length, could be outside $[0, 1]$
- $r = t P'_2$ — vect to closest point on the segment (from P_1)
    - clamped(t) -> distance to edge points
- $o = P'_s - r$ — perpendicular from sample to line
- $d = |o|^2 = o \cdot o$ — squared distance from the sample to line


In [ ]:
def dist1_t(p1, p2, ps):
    ps_ = ps - p1
    p2_ = p2 - p1
    l = p2_ @ p2_
    t = (ps_ @ p2_) / l
    return min(1.0, max(0.0, t))


def disq1(p1, p2, t, ps):
    ps_ = ps - p1
    p2_ = p2 - p1
    o = ps_ - t * p2_
    return o @ o


def dist1map(p):
    t_ = [dist1_t(points2d[i], points2d[i + 1], p) for i in range(N - 1)]
    d_ = [disq1(points2d[i], points2d[i + 1], t_[i], p) for i in range(N - 1)]
    d_min = min(d_)
    return math.sqrt(d_min)

In [ ]:
values = np.fromiter(map(dist1map, PP), dtype=float)

In [ ]:
grid.cell_data["scalars"] = values.flatten("F")
_ = pl.add_mesh(grid, name="tex", cmap="seismic", clim=[-1.0, +1.0], interpolate_before_map=False)